# Orbit Branch Layer Tests

Validation notebook for `orbit_branch.py`, the independent orbit branch runner/comparator layer.


## 1. Imports and setup

The notebook is intended to run from `Dev/05_Orbit_Branches/`. Generated files stay inside `./orbit_branch_tests/`.


In [1]:
from pathlib import Path
import shutil
import sys

import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from correctors import default_hd_corrector_kicks_rad
from cycle_time import RCSRamp
from errors import write_error_table, read_error_table
from machine_state import MachineState
from orbit_branch import (
    OrbitBranch,
    OrbitBranchConfig,
    OrbitBranchResult,
    compare_orbit_results,
    extract_orbit_df,
    summarise_orbit_df,
    summarise_orbit_difference,
)

LATTICE_FOLDER = "../Lattice_Files/00_Simplified_Lattice/"
SEQUENCE_NAME = "synchrotron"
OUTPUT_DIR = Path("./orbit_branch_tests")

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Output directory:", OUTPUT_DIR.resolve())


Project directory: /home/hr/Repositories/optics_gui/Dev/05_Orbit_Branches
Output directory: /home/hr/Repositories/optics_gui/Dev/05_Orbit_Branches/orbit_branch_tests


## 2. Shared machine state

Build a reproducible base machine state at one cycle time. Branches may write their own machine-state files from this object.


In [2]:
ramp = RCSRamp(top_energy_MeV=800.0)
beam_state = ramp.state_at(3.0)

base_machine_state = MachineState.from_defaults(
    beam_state=beam_state,
    main_magnet_mode="srm_bare",
    requested_qx=4.295,
    requested_qy=3.825,
    tune_method="di_wright",
)

assert base_machine_state.kqtd != 0.0
assert base_machine_state.kqtf != 0.0
print(base_machine_state.summary_dict()["cycle_time_ms"])


3.0


## 3. Default branch

The default branch should run an isolated MAD-X model, write a branch-local machine-state file, and return TWISS, orbit and summary DataFrames.


In [3]:
default_branch = OrbitBranch(
    OrbitBranchConfig(
        name="default",
        lattice_folder=LATTICE_FOLDER,
        sequence_name=SEQUENCE_NAME,
        machine_state=base_machine_state,
        aperture_file="ISIS.aperture",
        output_dir=OUTPUT_DIR / "default",
        metadata={"kind": "default"},
    )
)

default_result = default_branch.run()

assert isinstance(default_result, OrbitBranchResult)
assert default_result.name == "default"
assert Path(default_result.machine_state_file).is_file()
assert default_result.twiss_df is not default_result.orbit_df
assert not default_result.twiss_df.empty
assert not default_result.orbit_df.empty
assert not default_result.summary_df.empty
assert {"name", "keyword", "s", "x", "y", "x_mm", "y_mm", "orbit_radius_mm"}.issubset(default_result.orbit_df.columns)
assert default_result.metadata["branch_name"] == "default"
assert default_result.metadata["kind"] == "default"
assert default_result.metadata["errors_applied"] is False
assert default_result.metadata["orbit_summary"]["n_rows"] == len(default_result.orbit_df)
assert default_result.metadata["loaded_files"]

orbit_summary = summarise_orbit_df(default_result.orbit_df)
assert orbit_summary["n_rows"] == len(default_result.orbit_df)
assert orbit_summary["max_orbit_radius_mm"] == 0.0

default_result.orbit_df.head()


,name,keyword,s,x,y,px,py,x_mm,y_mm,orbit_radius_m,orbit_radius_mm
#s,synchrotron$start:1,marker,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
sp0_datum,sp0_datum:1,marker,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
sp0_dipfr8,sp0_dipfr8:1,sbend,0.16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
sp0_dipfr9,sp0_dipfr9:1,sbend,0.36,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
sp0_dipfr10,sp0_dipfr10:1,sbend,0.39,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 4. Misaligned branch

A synthetic single-element error table should apply through the branch and produce observable orbit deltas versus the default branch.


In [4]:
misalignment_table = pd.DataFrame(
    [
        {
            "name": "sp0_qd:1",
            "dx": 1.0e-3,
            "dy": 3.0e-4,
            "ds": 0.0,
            "dphi": 0.0,
            "dtheta": 0.0,
            "dpsi": 0.0,
        }
    ]
)
misalignment_path = write_error_table(
    misalignment_table,
    OUTPUT_DIR / "misalignment_qd.tfs",
    table_name="MISALIGNMENT_QD",
)

misaligned_result = OrbitBranch(
    OrbitBranchConfig(
        name="misaligned_qd",
        lattice_folder=LATTICE_FOLDER,
        sequence_name=SEQUENCE_NAME,
        machine_state=base_machine_state,
        aperture_file="ISIS.aperture",
        error_table_paths=[misalignment_path],
        output_dir=OUTPUT_DIR / "misaligned_qd",
        metadata={"kind": "misaligned"},
    )
).run()

assert misaligned_result.metadata["errors_applied"] is True
assert misalignment_path in misaligned_result.metadata["loaded_files"]
assert misaligned_result.error_table_paths == [misalignment_path]

misaligned_comparison = compare_orbit_results(default_result, misaligned_result)
misaligned_summary = summarise_orbit_difference(misaligned_comparison)

assert 1.0 <= misaligned_summary["max_delta_orbit_radius_mm"] <= 20.0, misaligned_summary
assert misaligned_summary["n_rows"] == len(default_result.orbit_df)
misaligned_summary


{'n_rows': 586,
 'max_abs_delta_x_m': 0.002292551768992937,
 'max_abs_delta_y_m': 0.001562732040130335,
 'rms_delta_x_m': 0.0010974117820454455,
 'rms_delta_y_m': 0.0008201225450717856,
 'max_delta_orbit_radius_m': 0.00246748845785235,
 'max_abs_delta_x_mm': 2.292551768992937,
 'max_abs_delta_y_mm': 1.562732040130335,
 'rms_delta_x_mm': 1.0974117820454454,
 'rms_delta_y_mm': 0.8201225450717856,
 'max_delta_orbit_radius_mm': 2.46748845785235}

## 5. Corrector branch

A branch with nonzero corrector settings supplied through `MachineState` should alter the orbit without using error tables.


In [5]:
hd_kicks = default_hd_corrector_kicks_rad()
hd_kicks["r0hd1_kick"] = 1.0e-3

corrector_machine_state = MachineState.from_defaults(
    beam_state=beam_state,
    main_magnet_mode="srm_bare",
    requested_qx=4.295,
    requested_qy=3.825,
    tune_method="di_wright",
    hd_corrector_kicks_rad=hd_kicks,
    corrector_prefer="kicks",
)

corrector_result = OrbitBranch(
    OrbitBranchConfig(
        name="corrector_r0hd1",
        lattice_folder=LATTICE_FOLDER,
        sequence_name=SEQUENCE_NAME,
        machine_state=corrector_machine_state,
        aperture_file="ISIS.aperture",
        output_dir=OUTPUT_DIR / "corrector_r0hd1",
        metadata={"kind": "corrector"},
    )
).run()

corrector_comparison = compare_orbit_results(default_result, corrector_result)
corrector_summary = summarise_orbit_difference(corrector_comparison)

assert corrector_result.metadata["errors_applied"] is False
assert corrector_summary["max_abs_delta_x_mm"] > 1.0, corrector_summary
assert corrector_summary["max_abs_delta_y_mm"] == 0.0
corrector_summary


{'n_rows': 586,
 'max_abs_delta_x_m': 0.008011170963020089,
 'max_abs_delta_y_m': 0.0,
 'rms_delta_x_m': 0.003914051708342724,
 'rms_delta_y_m': 0.0,
 'max_delta_orbit_radius_m': 0.008011170963020089,
 'max_abs_delta_x_mm': 8.011170963020088,
 'max_abs_delta_y_mm': 0.0,
 'rms_delta_x_mm': 3.9140517083427238,
 'rms_delta_y_mm': 0.0,
 'max_delta_orbit_radius_mm': 8.011170963020088}

## 6. Multiple error tables

Multiple error tables should be combined into one branch-local SETERR table before MAD-X application.


In [6]:
error_dx_path = write_error_table(
    pd.DataFrame([
        {"name": "sp0_qf:1", "dx": 3.0e-4, "dy": 0.0, "ds": 0.0, "dphi": 0.0, "dtheta": 0.0, "dpsi": 0.0}
    ]),
    OUTPUT_DIR / "qf_dx.tfs",
    table_name="QF_DX",
)
error_dy_path = write_error_table(
    pd.DataFrame([
        {"name": "sp0_qf:1", "dx": 0.0, "dy": 1.0e-3, "ds": 0.0, "dphi": 0.0, "dtheta": 0.0, "dpsi": 0.0}
    ]),
    OUTPUT_DIR / "qf_dy.tfs",
    table_name="QF_DY",
)

combined_result = OrbitBranch(
    OrbitBranchConfig(
        name="combined_qf_errors",
        lattice_folder=LATTICE_FOLDER,
        sequence_name=SEQUENCE_NAME,
        machine_state=base_machine_state,
        aperture_file="ISIS.aperture",
        error_table_paths=[error_dx_path, error_dy_path],
        output_dir=OUTPUT_DIR / "combined_qf_errors",
    )
).run()

assert len(combined_result.error_table_paths) == 1
combined_path = Path(combined_result.error_table_paths[0])
assert combined_path.is_file()
assert combined_path.parent.name == "error_tables"

combined_table = read_error_table(combined_path)
combined_row = combined_table.loc[combined_table["name"] == "sp0_qf:1"].iloc[0]
assert np.isclose(combined_row["dx"], 3.0e-4)
assert np.isclose(combined_row["dy"], 1.0e-3)

combined_summary = summarise_orbit_difference(compare_orbit_results(default_result, combined_result))
assert 1.0 <= combined_summary["max_delta_orbit_radius_mm"] <= 20.0, combined_summary
combined_summary


{'n_rows': 586,
 'max_abs_delta_x_m': 0.0010227102411198862,
 'max_abs_delta_y_m': 0.003203126956117319,
 'rms_delta_x_m': 0.00048602096179471406,
 'rms_delta_y_m': 0.001673826263494676,
 'max_delta_orbit_radius_m': 0.003232707113271795,
 'max_abs_delta_x_mm': 1.0227102411198863,
 'max_abs_delta_y_mm': 3.2031269561173192,
 'rms_delta_x_mm': 0.48602096179471405,
 'rms_delta_y_mm': 1.673826263494676,
 'max_delta_orbit_radius_mm': 3.232707113271795}

## 7. Comparison and isolation checks

Comparisons should preserve row order and independent branches should use distinct output/model directories.


In [7]:
assert default_result.metadata["output_dir"] != misaligned_result.metadata["output_dir"]
assert default_result.metadata["output_dir"] != corrector_result.metadata["output_dir"]
assert default_result.metadata["output_dir"] != combined_result.metadata["output_dir"]

comparison = compare_orbit_results(default_result.orbit_df, misaligned_result.orbit_df)
assert {"delta_x", "delta_y", "delta_x_mm", "delta_y_mm", "delta_orbit_radius_mm"}.issubset(comparison.columns)
assert comparison["name"].tolist() == default_result.orbit_df["name"].tolist()

summary = summarise_orbit_difference(comparison)
assert summary["max_delta_orbit_radius_mm"] == misaligned_summary["max_delta_orbit_radius_mm"]
comparison.head()


,name,keyword,s,x_reference,y_reference,x_other,y_other,delta_x,delta_y,delta_orbit_radius_m,delta_x_mm,delta_y_mm,delta_orbit_radius_mm
0,synchrotron$start:1,marker,0.00,0.0,0.0,-0.001567,-0.000590,-0.001567,-0.000590,0.001674,-1.567033,-0.589882,1.674382
1,sp0_datum:1,marker,0.00,0.0,0.0,-0.001567,-0.000590,-0.001567,-0.000590,0.001674,-1.567033,-0.589882,1.674382
2,sp0_dipfr8:1,sbend,0.16,0.0,0.0,-0.001535,-0.000627,-0.001535,-0.000627,0.001658,-1.534677,-0.627014,1.657824
3,sp0_dipfr9:1,sbend,0.36,0.0,0.0,-0.001494,-0.000674,-0.001494,-0.000674,0.001639,-1.493932,-0.673531,1.638742
4,sp0_dipfr10:1,sbend,0.39,0.0,0.0,-0.001488,-0.000681,-0.001488,-0.000681,0.001636,-1.487824,-0.680506,1.636065


## 8. Negative tests

Bad inputs should fail before silent plotting or GUI misuse.


In [8]:
try:
    extract_orbit_df(pd.DataFrame({"name": ["a"], "x": [0.0]}))
    raise AssertionError("Expected extract_orbit_df missing-column failure")
except ValueError as exc:
    print("Caught expected extract_orbit_df error:", exc)

try:
    summarise_orbit_df(pd.DataFrame({"x": [0.0], "y": [0.0]}))
    raise AssertionError("Expected summarise_orbit_df missing-column failure")
except ValueError as exc:
    print("Caught expected summarise_orbit_df error:", exc)

try:
    compare_orbit_results(default_result.orbit_df.iloc[:-1], misaligned_result.orbit_df)
    raise AssertionError("Expected row-count comparison failure")
except ValueError as exc:
    print("Caught expected comparison row-count error:", exc)

renamed_orbit = misaligned_result.orbit_df.copy()
renamed_orbit.loc[renamed_orbit.index[0], "name"] = "different_name"
try:
    compare_orbit_results(default_result.orbit_df, renamed_orbit)
    raise AssertionError("Expected name mismatch comparison failure")
except ValueError as exc:
    print("Caught expected comparison name error:", exc)

try:
    OrbitBranch("not a config")
    raise AssertionError("Expected OrbitBranch config type failure")
except TypeError as exc:
    print("Caught expected config type error:", exc)

try:
    OrbitBranch(
        OrbitBranchConfig(
            name="missing_error_table",
            lattice_folder=LATTICE_FOLDER,
            sequence_name=SEQUENCE_NAME,
            machine_state=base_machine_state,
            aperture_file="ISIS.aperture",
            error_table_paths=[OUTPUT_DIR / "missing_error_table.tfs"],
            output_dir=OUTPUT_DIR / "missing_error_table",
        )
    ).run()
    raise AssertionError("Expected missing error-table failure")
except FileNotFoundError as exc:
    print("Caught expected missing error-table error:", exc)

print("Negative tests passed")


Caught expected extract_orbit_df error: TWISS DataFrame is missing orbit columns: ['keyword', 's', 'y', 'px', 'py']
Caught expected summarise_orbit_df error: Orbit DataFrame is missing summary columns: ['orbit_radius_m']
Caught expected comparison row-count error: Orbit DataFrames have different row counts.
Caught expected comparison name error: Orbit DataFrames have different row order or element names.
Caught expected config type error: config must be an OrbitBranchConfig.
Caught expected missing error-table error: Missing MAD-X error table: orbit_branch_tests/missing_error_table.tfs
Negative tests passed


## 9. Final status


In [9]:
print("orbit_branch.py layer validation complete")


orbit_branch.py layer validation complete
